# ClimateBERT on climate news vs CCAM

[`climatebert/distilroberta-base-climate-f`](https://huggingface.co/climatebert/distilroberta-base-climate-f) is a **base language model** (Masked LM) further pretrained on climate text. It is **not** itself a sentiment/QA classifier.

We use the official **fine-tuned heads built from that base** (still no training on our side):

| Model | Labels |
|---|---|
| `distilroberta-base-climate-sentiment` | opportunity / neutral / risk |
| `distilroberta-base-climate-commitment` | commitment/action yes/no |
| `distilroberta-base-climate-detector` | climate-related yes/no |

Run on the same year-stratified sample as BERT QA:
```bash
python analysis/climatebert/run_classifiers.py --reuse-qa-sample
# or a fresh sample:
python analysis/climatebert/run_classifiers.py --per-year 40
```

**Mapping to CCAM (approximate):** risk share ↔ worry / harm; commitment share ↔ policy support / clean-energy priority. News framing ≠ public opinion.

In [ ]:
%pip install pandas pyarrow matplotlib seaborn numpy transformers torch -q

In [ ]:
from pathlib import Path
import sys
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT / "analysis" / "shared"))
from ccam_prepare import apply_recodes, wave_level_outcomes

sns.set_theme(style="whitegrid", context="notebook")
CB_DIR = Path(".")
CCAM_PATH = ROOT / "ccam" / "CCAM_SPSS_Data_2008-2025.csv"

In [ ]:
preds_path = CB_DIR / "climatebert_predictions.parquet"
yearly_path = CB_DIR / "climatebert_yearly_rates.parquet"
if not preds_path.exists():
    raise FileNotFoundError("Run: python analysis/climatebert/run_classifiers.py --reuse-qa-sample")

preds = pd.read_parquet(preds_path)
yearly = pd.read_parquet(yearly_path)
meta = json.loads((CB_DIR / "climatebert_run_meta.json").read_text())
print(json.dumps(meta, indent=2))
print(f"articles: {len(preds):,}")
display(preds[["year", "title", "sentiment_label", "commitment_label", "detector_label"]].head())
display(preds["sentiment_label"].value_counts(normalize=True).rename("share").to_frame().round(3))

## ClimateBERT rates over time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(yearly["year"], yearly["risk_rate"], marker="o", label="sentiment = risk")
axes[0].plot(yearly["year"], yearly["opportunity_rate"], marker="o", label="sentiment = opportunity")
axes[0].plot(yearly["year"], yearly["neutral_rate"], marker="o", label="sentiment = neutral")
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("Share of articles")
axes[0].set_title("ClimateBERT sentiment classes in news sample")
axes[0].legend(fontsize=8)

axes[1].plot(yearly["year"], yearly["commitment_rate"], marker="o", label="commitment/action = yes")
axes[1].plot(yearly["year"], yearly["climate_related_rate"], marker="o", label="detector = climate-related")
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("Share of articles")
axes[1].set_xlabel("Year")
axes[1].set_title("ClimateBERT commitment & climate-detector rates")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()
display(yearly.round(3).tail())

## Compare to CCAM

In [ ]:
ccam = apply_recodes(pd.read_csv(CCAM_PATH, low_memory=False))
wave_out = wave_level_outcomes(ccam)

def weighted_share(df, col, values):
    s, w = df[col], df["weight_wave"]
    mask = s.notna() & w.notna() & ~s.astype(str).str.lower().eq("refused")
    if mask.sum() == 0:
        return float("nan")
    return float((w[mask] * s[mask].astype(str).isin(values).astype(float)).sum() / w[mask].sum())

belief_years = []
for year, g in ccam.groupby("year"):
    belief_years.append(
        {
            "year": int(year),
            "happening_yes": weighted_share(g, "happening", {"Yes"}),
            "human_caused": weighted_share(
                g, "cause_recoded", {"Caused mostly by human activities"}
            ),
        }
    )
ccam_year = (
    wave_out.assign(year=wave_out["wave_date"].dt.year)
    .groupby("year", as_index=False)[
        ["worry_high", "harm_US_high", "support_CO2_reg", "priority_cleanenergy_high", "discuss_often"]
    ]
    .mean()
    .merge(pd.DataFrame(belief_years), on="year", how="outer")
    .sort_values("year")
)

comparisons = [
    ("risk_rate", "worry_high", "ClimateBERT risk vs CCAM worry"),
    ("risk_rate", "harm_US_high", "ClimateBERT risk vs CCAM US harm"),
    ("opportunity_rate", "priority_cleanenergy_high", "ClimateBERT opportunity vs clean-energy priority"),
    ("commitment_rate", "support_CO2_reg", "ClimateBERT commitment vs CO₂ regulation support"),
    ("commitment_rate", "priority_cleanenergy_high", "ClimateBERT commitment vs clean-energy priority"),
]

fig, axes = plt.subplots(3, 2, figsize=(12, 9), sharex=True)
axes = axes.ravel()
corr_rows = []
for ax, (cb_col, ccam_col, title) in zip(axes, comparisons):
    m = yearly[["year", cb_col]].merge(ccam_year[["year", ccam_col]], on="year", how="inner").dropna()
    ax.plot(m["year"], m[cb_col], marker="o", label=f"ClimateBERT {cb_col}")
    ax.plot(m["year"], m[ccam_col], marker="s", label=f"CCAM {ccam_col}")
    ax.set_ylim(0, 1)
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=7)
    r = m[cb_col].corr(m[ccam_col]) if len(m) >= 5 else float("nan")
    corr_rows.append({"climatebert": cb_col, "ccam": ccam_col, "n_years": len(m), "pearson_r": r})

# hide unused last axis if odd count
if len(comparisons) < len(axes):
    for ax in axes[len(comparisons) :]:
        ax.axis("off")

fig.suptitle("ClimateBERT news signals vs CCAM public opinion", y=1.01)
plt.tight_layout()
plt.show()

display(pd.DataFrame(corr_rows).style.format({"pearson_r": "{:.3f}"}))

In [ ]:
# Optional: side-by-side with BERT-QA affirm rates if that run exists
qa_yearly = Path("..") / "bert_qa" / "qa_yearly_affirm_rates.parquet"
if qa_yearly.exists():
    qa = pd.read_parquet(qa_yearly)
    risk = yearly[["year", "risk_rate"]]
    worry_qa = qa[qa["item"] == "public_worry"][["year", "affirm_rate"]].rename(
        columns={"affirm_rate": "bert_qa_worry_affirm"}
    )
    m = risk.merge(worry_qa, on="year", how="inner").merge(
        ccam_year[["year", "worry_high"]], on="year", how="inner"
    )
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(m["year"], m["risk_rate"], marker="o", label="ClimateBERT risk")
    ax.plot(m["year"], m["bert_qa_worry_affirm"], marker="o", label="BERT-QA worry affirm")
    ax.plot(m["year"], m["worry_high"], marker="s", label="CCAM worry")
    ax.set_ylim(0, 1)
    ax.set_title("Three views of 'concern': ClimateBERT risk, BERT-QA, CCAM")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("No bert_qa yearly file yet — skip three-way plot.")